In [0]:
%sql
use catalog :catalog_name

- Load raw CSV
- Append-only
- Add metadata

In [0]:
from pyspark.sql.functions import *

# Get parameters from widgets
catalog_name = dbutils.widgets.get("catalog_name")


# Paths
base_path = f"/Volumes/23283_dev/default/23283_datasets/"

def load_bronze(table_name, file_path):
    df = spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(file_path)

    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file", col("_metadata.file_path"))

    df.write.format("delta") \
        .mode("append") \
        .saveAsTable(f"{bronze_db}.{table_name}")

# Load all datasets
load_bronze("customers", f"{base_path}customers.csv")
load_bronze("products", f"{base_path}products.csv")
load_bronze("orders", f"{base_path}orders.csv")
load_bronze("order_items", f"{base_path}order_items.csv")